In [ ]:
FORCE_CPU = False # set to True to force CPU execution

try:
    if FORCE_CPU:
        raise ImportError
    import cupy as xp
    if not xp.cuda.is_available():
        raise ImportError
    xp.matmul(xp.zeros((1, 1)), xp.zeros((1, 1)))
    using_gpu = True # True for CuPy (GPU), False for NumPy (CPU)
except Exception:
    import numpy as xp
    using_gpu = False # True for CuPy (GPU), False for NumPy (CPU)
import pandas

In [ ]:
import os

csv_path = 'mnist.csv'
if not os.path.exists(csv_path):
    fallback_path = os.path.join('MNIST Digit Recogniser', 'mnist.csv')
    if os.path.exists(fallback_path):
        csv_path = fallback_path
    else:
        raise FileNotFoundError("Error: mnist.csv not found.")

data = xp.array(pandas.read_csv(csv_path))

xp.random.shuffle(data)

# first 2000 for testing
test_data = data[:2000]

# next 4000 for early stopping
dev_data = data[2000:6000]

# remaining for training
train = data[6000:]

datapoints = train.shape[0]
pixels = train.shape[1] - 1

labels = train[:, 0]
pics = (train[:, 1:] / 255).T

testDatapoints = test_data.shape[0]
testLabels = test_data[:, 0]
testPics = (test_data[:, 1:] / 255).T

devDatapoints = dev_data.shape[0]
devLabels = dev_data[:, 0]
devPics = (dev_data[:, 1:] / 255).T


In [ ]:
one_hot = xp.zeros((datapoints, 10), dtype = int)
one_hot[xp.arange(datapoints), labels] = 1
one_hot = one_hot.T

dev_hot = xp.zeros((devDatapoints, 10), dtype = int)
dev_hot[xp.arange(devDatapoints), devLabels] = 1
dev_hot = dev_hot.T

In [ ]:
layers = [pixels, 16, 16, 10] # dont touch first and last entries, change rest to modify hidden layers

weights = list()
biases = list()

for i in range(len(layers)):
    weights.append(xp.random.randn(layers[i], layers[i - 1]) * xp.sqrt(1 / layers[i - 1])) # xavier initialisation by scaling normal distribution
    biases.append(xp.zeros((layers[i], 1)))

In [ ]:
def ReLU(arr):
    ret = arr.copy()
    mask = ret < 0
    ret[mask] = 0
    return ret

In [ ]:
def ReLU_prime(arr):
    ret = xp.zeros(arr.shape)
    mask = arr > 0
    ret[mask] = 1
    return ret

In [ ]:
def softmax(arr):
    ret = arr.copy()
    ret -= xp.max(ret, axis = 0, keepdims = True) # overflow safety
    denom = xp.sum(xp.exp(ret), axis = 0)
    ret = xp.exp(ret) / denom
    return ret

In [ ]:
epochs = 10000 # max epochs
learningRate = .1
epsilon = 1e-10 # prevent log(0) domain error
batch_size = 128

# for early stopping

fails = 0
patience = 25
bestLoss = float("inf")
recordWs = list()
recordBs = list()

In [ ]:
for i in range(epochs):
    
    perm = xp.random.permutation(datapoints)
    pics = pics[:, perm]
    one_hot = one_hot[:, perm]

    # mini-batching

    for k in range(0, datapoints, batch_size):
        batchPics = pics[:, k:k + batch_size]
        batchHot = one_hot[:, k:k + batch_size]
        currentBatchSize = batchPics.shape[1] # datapoints / batchSize may not be an int

        # forward propagation

        Zs = [batchPics,]
        As = [batchPics,]

        for j in range(1, len(layers)):
            Z = xp.matmul(weights[j], As[j - 1]) + biases[j]
            Zs.append(Z)

            A = ReLU(Z)
            As.append(A)

        As.pop()
        pred = softmax(Zs[-1])
        As.append(pred)

        # backwards propagation

        deltaZ = pred - batchHot

        deltaWs = list()
        deltaBs = list()

        for j in range(len(layers) - 1, 0, -1):
            deltaW = xp.matmul(deltaZ, As[j - 1].T) / currentBatchSize
            deltaWs.append(deltaW)

            deltaB = xp.sum(deltaZ, axis = 1, keepdims = True) / currentBatchSize
            deltaBs.append(deltaB)

            if j > 1:
                deltaZ = xp.matmul(weights[j].T, deltaZ) * ReLU_prime(Zs[j  -1])

        deltaWs.reverse()
        deltaBs.reverse()

        # learning

        for j in range(1, len(layers)):
            weights[j] = weights[j] - learningRate * deltaWs[j - 1]
            biases[j] = biases[j] - learningRate * deltaBs[j - 1]

    # early stopping

    Zs = [devPics,]
    As = [devPics,]

    for j in range(1, len(layers)):
        Z = xp.matmul(weights[j], As[j - 1]) + biases[j]
        Zs.append(Z)

        A = ReLU(Z)
        As.append(A)

    As.pop()
    pred = softmax(Zs[-1])
    As.append(pred)

    L = (-1 / devDatapoints) * xp.sum(dev_hot * xp.log(pred + epsilon))
    
    if bestLoss > L:
        bestLoss = L
        fails = 0
        recordWs = [w.copy() for w in weights]
        recordBs = [b.copy() for b in biases]
    else:
        fails += 1

    if fails > patience:
        print("Epochs trained: ", i)
        break

else:
    print(f"Epochs trained: {epochs}")

In [ ]:
Zs = [testPics,]
As = [testPics,]

for j in range(1, len(layers)):
    Z = xp.matmul(recordWs[j], As[j - 1]) + recordBs[j]
    Zs.append(Z)

    A = ReLU(Z)
    As.append(A)

As.pop()
pred = softmax(Zs[-1])

predictions = xp.argmax(pred, axis = 0)
correct = xp.sum(predictions == testLabels)

print("Accuracy: " + str(round((float(correct) / testDatapoints) * 100, 3)) + "%")

In [ ]:
from matplotlib import pyplot as plt

def visualise_prediction(index):
    if using_gpu:
        image = testPics[:, index].get().reshape(28, 28)
        probabilities = pred[:, index].get()
        network_guess = int(predictions[index].get())
        true_answer = int(testLabels[index].get())
    else:
        image = testPics[:, index].reshape(28, 28)
        probabilities = pred[:, index]
        network_guess = int(predictions[index])
        true_answer = int(testLabels[index])
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize = (10, 4))
    
    ax1.imshow(image, cmap='gray')
    ax1.set_title(f"True Label: {true_answer} | Model Guessed: {network_guess}", fontsize = 14)
    ax1.axis('off')

    bars = ax2.bar(range(10), probabilities, color = "gray")
    ax2.set_xticks(range(10))
    ax2.set_ylim([0, 1.1])
    ax2.set_xlabel("Digit", fontsize = 12)
    ax2.set_ylabel("Probability", fontsize = 12)
    ax2.set_title("Model's Internal Confidence", fontsize = 14)
    
    bars[network_guess].set_color("red") 
    
    bars[true_answer].set_color("green")
    
    plt.tight_layout()
    plt.show()

for i in range(50): # adjust displayed images range
    visualise_prediction(i)
